# 12b · Train a fast encoder decision model

A recorded trace ends before the expected outcome was verified. Should the desk classify it as a premature stop, a missing verification step, or another kind of failure? Compare a few-shot encoder, hard supervision, teacher distributions and a hybrid repair. Treat tool calls as recorded data throughout.

[Open the lesson](https://docs.recohut.com/courses/advanced-llm-fine-tuning/3-adapt-models/12b-train-a-fast-encoder-decision-model).

## Which environment runs the comparison?

Use a single T4. The encoder arms use FP16 automatic mixed precision and the generative comparator uses FP16 weights. The T4 lacks native BF16 Tensor Cores. BF16 performance is outside this comparison.

In [1]:
import os
import subprocess
import sys

os.environ["USE_TF"] = "0"
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.57.1", "sentence-transformers==5.1.2",
    "scikit-learn==1.7.2", "huggingface-hub==0.36.0", "pyarrow==20.0.0"], check=True)

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'transformers==4.57.1', 'sentence-transformers==5.1.2', 'scikit-learn==1.7.2', 'huggingface-hub==0.36.0', 'pyarrow==20.0.0'], returncode=0)

## Which functions define the experiment?

Keep data preparation, class definitions, model heads, training and scoring explicit. Teacher and reference labels use the same option order.

In [2]:
"""Data preparation, training and scoring for the trace triage experiment."""

import hashlib
import json
import math
import random
import re
from collections import Counter
from pathlib import Path
from time import perf_counter

import numpy as np
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, f1_score
from sklearn.pipeline import make_pipeline
from torch import nn
from torch.nn import functional as F

SEED = 42
FAILURES = {
    "incorrect_tool_selection": "The selected tool does a different job than the current step requires.",
    "invalid_tool_arguments": "A supplied argument has an invalid type or value.",
    "missing_required_arguments": "A required argument is absent from the call.",
    "hallucinated_tool_output": "A claim invents a result that the tool never returned.",
    "failure_to_follow_constraints": "An action violates an explicit request constraint.",
    "premature_completion": "The workflow stops while required work remains.",
    "failure_to_recover": "A recoverable tool error is present and the workflow fails to handle it.",
    "failure_to_verify_completion": "The workflow skips a needed check that the intended outcome happened.",
    "incorrect_final_answer": "The final answer gives a wrong result despite the available information.",
}
OPTIONS = {
    **FAILURES,
    "clean": "The recorded workflow completed without an annotated failure.",
    "out_of_scope": "A standalone request falls outside the trace-triage queue.",
}
CLASSES = tuple(OPTIONS)
BLIND_SPOTS = {"failure_to_verify_completion": "premature_completion",
               "hallucinated_tool_output": "incorrect_final_answer"}
SPLITS = ("train", "gold_val", "gold_calibration", "gold_test")
COUNTS = {"train": 32, "gold_val": 12, "gold_calibration": 12, "gold_test": 16}
MAX_LENGTH = 384


def norm(text):
    return " ".join(re.findall(r"\w+", text.casefold()))


def stable_id(value):
    return hashlib.sha256(value.encode()).hexdigest()


def trace_example(row):
    category = row["failure_category"] or "clean"
    if category not in OPTIONS:
        return None
    # Inputs available in an ordinary replay. Omit all diagnosis and correction fields.
    steps = [
        {"call": step.get("tool_call"), "result": step.get("tool_output")}
        for step in row["agent_trajectory"]
    ]
    body = {"request": row["task_description"], "constraints": row["user_constraints"],
            "observations": steps, "reply": row["final_answer"]}
    text = json.dumps(body, ensure_ascii=False, sort_keys=True)
    return {"id": row["task_id"], "group": row["scenario_id"],
            "domain": row["domain"], "text": text, "label": category,
            "source": "AgentFailureBench"}


def prepare_splits(trace_rows, oos):
    traces = [item for row in trace_rows if (item := trace_example(row)) is not None]
    # The card reports one canonical task template per domain. Keep domains intact.
    domains = sorted({item["domain"] for item in traces})
    assert len(domains) >= 7, domains
    random.Random(SEED).shuffle(domains)
    assignment = dict(zip(SPLITS, (domains[4:], domains[:1], domains[1:2], domains[2:4])))
    groups, unique_texts, ids = set(), set(), set()
    splits = {}
    for split, domains_for_split in assignment.items():
        pools = [item for item in traces if item["domain"] in domains_for_split]
        pools += [{"id": "oos:" + stable_id(norm(text)), "group": "oos:" + stable_id(norm(text)),
                   "domain": "none", "text": text, "label": "out_of_scope",
                   "source": "CLINC150"} for text in oos[split]]
        selected, counts = [], Counter()
        for row in sorted(pools, key=lambda row: stable_id(str(SEED) + split + row["id"])):
            signature = norm(row["text"])
            if counts[row["label"]] >= COUNTS[split]:
                continue
            if signature in unique_texts or row["group"] in groups or row["id"] in ids:
                continue
            selected.append(row)
            counts[row["label"]] += 1
            unique_texts.add(signature)
            groups.add(row["group"])
            ids.add(row["id"])
        splits[split] = selected
    return splits, assignment


def label_ids(rows):
    return torch.tensor([CLASSES.index(row["label"]) for row in rows], dtype=torch.long)


def macro_score(labels, predictions):
    return float(f1_score(labels, predictions, labels=range(len(CLASSES)), average="macro", zero_division=0))


def teacher_fit(train):
    # A weak, lexical teacher. We collapse two classes to represent known gaps in its rubric.
    teacher = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), max_features=10000),
                            LogisticRegression(max_iter=500, random_state=SEED))
    teacher.fit([r["text"] for r in train], [BLIND_SPOTS.get(r["label"], r["label"]) for r in train])
    return teacher


def teacher_probs(teacher, rows):
    raw = teacher.predict_proba([r["text"] for r in rows])
    out = np.zeros((len(rows), len(CLASSES)), dtype=np.float32)
    for j, name in enumerate(teacher.classes_):
        out[:, CLASSES.index(name)] = raw[:, j]
    return torch.tensor(out)


def per_class(labels, predictions):
    matrix = confusion_matrix(labels, predictions, labels=range(len(CLASSES)))
    return {"confusion": matrix.tolist(), "macro_f1": macro_score(labels, predictions),
            "recall": {name: float(matrix[i, i] / matrix[i].sum()) if matrix[i].sum() else None
                       for i, name in enumerate(CLASSES)},
            "counts": {name: int(matrix[i].sum()) for i, name in enumerate(CLASSES)}}


def soft_ce(target, logits):
    return -(target * torch.log_softmax(logits.float(), -1)).sum(-1).mean()


def hybrid_targets(target, labels, overrides):
    mask = torch.isin(labels, torch.tensor(overrides, dtype=labels.dtype, device=labels.device))
    gold = F.one_hot(labels, num_classes=target.shape[-1]).float()
    return torch.where(mask.unsqueeze(-1), gold, target)


def reliability(logits, labels, temperature=1.0, bins=10):
    confidence, predictions = (logits.float() / temperature).softmax(-1).max(-1)
    correct = predictions.eq(labels).float()
    groups = torch.clamp((confidence * bins).long(), max=bins - 1)
    rows = []
    for i in range(bins):
        mask = groups.eq(i)
        n = int(mask.sum())
        rows.append(dict(count=n, low=i / bins, high=(i + 1) / bins,
                         confidence=float(confidence[mask].mean()) if n else None,
                         accuracy=float(correct[mask].mean()) if n else None))
    ece = sum(r["count"] / len(labels) * abs(r["confidence"] - r["accuracy"]) for r in rows if r["count"])
    return {"ece": ece, "nll": float(F.cross_entropy(logits.float() / temperature, labels)), "bins": rows}


def fit_temperature(logits, labels):
    # The optimisation can never see the gold test labels.
    lower, upper = math.log(0.05), math.log(20.0)
    for _ in range(40):
        left = (2 * lower + upper) / 3
        right = (lower + 2 * upper) / 3
        if F.cross_entropy(logits / math.exp(left), labels) <= F.cross_entropy(logits / math.exp(right), labels):
            upper = right
        else:
            lower = left
    return math.exp((lower + upper) / 2)


class OptionScorer(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        hidden = encoder.config.hidden_size
        self.head = nn.Sequential(nn.Linear(hidden * 4, hidden), nn.GELU(), nn.Linear(hidden, 1))

    def encode(self, features):
        last = self.encoder(**features).last_hidden_state
        mask = features["attention_mask"].unsqueeze(-1).to(last.dtype)
        return (last * mask).sum(1) / mask.sum(1).clamp_min(1)

    def forward(self, input_features, options_features):
        inputs = self.encode(input_features)
        options = self.encode(options_features)
        inputs = inputs[:, None, :].expand(-1, len(options), -1)
        options = options[None, :, :].expand(len(inputs), -1, -1)
        return self.head(torch.cat((inputs, options, inputs * options, (inputs - options).abs()), dim=-1)).squeeze(-1)


def encode_features(tokenizer, texts, max_length=MAX_LENGTH):
    tokenizer.truncation_side = "left"
    return tokenizer(texts, truncation=True, max_length=max_length,
                     padding=True, pad_to_multiple_of=8, return_token_type_ids=False, return_tensors="pt")


def device_features(features, device, indices=None):
    return {key: (value if indices is None else value[indices]).to(device)
            for key, value in features.items()}


def evaluate_options(model, features, options, device, batch_size=8):
    model.eval()
    batched = []
    with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
        for ids in torch.arange(len(features["input_ids"])).split(batch_size):
            logits = model(device_features(features, device, ids), options).float()
            assert torch.isfinite(logits).all()
            batched.append(logits.cpu())
    return torch.cat(batched)


def choose_checkpoints(history):
    gold_epoch = max(history, key=lambda row: (row["val_gold_macro_f1"], -row["epoch"]))["epoch"]
    teacher_epoch = max(history, key=lambda row: (row["val_teacher_agreement"], -row["epoch"]))["epoch"]
    return gold_epoch, teacher_epoch


def fit_options(new_encoder, features, options, labels, target, teacher_val, device, *, epochs=4, batch_size=8):
    # Reset model, batch order and head seed for every arm.
    torch.manual_seed(SEED)
    model = OptionScorer(new_encoder()).to(device)
    optimizer = torch.optim.AdamW([{"params": model.encoder.parameters(), "lr": 2e-5},
                                  {"params": model.head.parameters(), "lr": 1e-4}])
    scaler = torch.amp.GradScaler("cuda", init_scale=128, enabled=device.type == "cuda")
    order = torch.Generator().manual_seed(SEED)
    history = []
    best = {}
    for epoch in range(1, epochs + 1):
        model.train()
        updates = 0
        for ids in torch.randperm(len(target), generator=order).split(batch_size):
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
                logits = model(device_features(features["train"], device, ids), options)
                loss = soft_ce(target[ids].to(device), logits)
            assert torch.isfinite(loss)
            prev_scale = scaler.get_scale()
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            updates += int(scaler.get_scale() >= prev_scale)
        assert updates > 0, "Every optimiser step was skipped"
        val = evaluate_options(model, features["gold_val"], options, device).argmax(-1)
        history.append({"epoch": epoch, "attempted_updates": math.ceil(len(target) / batch_size),
                        "updates": updates, "val_gold_macro_f1": macro_score(labels["gold_val"], val),
                        "val_teacher_agreement": float(val.eq(teacher_val).float().mean())})
        for name, selected in zip(("gold", "teacher"), choose_checkpoints(history)):
            if selected == epoch:
                best[name] = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
    return model, history, best


def tune_sentence_embedder(model, rows, device, *, n_per_class=12, pairs_per_class=16):
    # SetFit's contrastive sentence-pair task with a logistic head.
    pools = [[r["text"] for r in rows if r["label"] == name][:n_per_class] for name in CLASSES]
    assert all(len(pool) >= 8 for pool in pools)
    rng = random.Random(SEED)
    pairs = []
    for i, pool in enumerate(pools):
        negatives = [item for j, other in enumerate(pools) if j != i for item in other]
        for _ in range(pairs_per_class):
            a, b = rng.sample(pool, 2)
            pairs.extend(((a, b, 1.0), (a, rng.choice(negatives), 0.0)))
    rng.shuffle(pairs)
    optimiser = torch.optim.AdamW(model.parameters(), lr=2e-5)
    scaler = torch.amp.GradScaler("cuda", init_scale=128, enabled=device.type == "cuda")
    model.train()
    updates = 0
    for start in range(0, len(pairs), 16):
        left, right, targets = zip(*pairs[start:start + 16])
        optimiser.zero_grad(set_to_none=True)
        with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
            first = model(device_features(model.tokenize(list(left)), device))["sentence_embedding"]
            second = model(device_features(model.tokenize(list(right)), device))["sentence_embedding"]
            cosine = F.cosine_similarity(first.float(), second.float())
            loss = F.mse_loss(cosine, torch.tensor(targets, device=device))
        assert torch.isfinite(loss)
        prev_scale = scaler.get_scale()
        scaler.scale(loss).backward()
        scaler.step(optimiser)
        scaler.update()
        updates += int(scaler.get_scale() >= prev_scale)
    assert updates > 0, "SetFit skipped every optimiser step"
    model.eval()
    x = [text for pool in pools for text in pool]
    y = [i for i, pool in enumerate(pools) for _ in pool]
    return LogisticRegression(max_iter=500, random_state=SEED).fit(model.encode(x), y)

## Which benchmark rows can reach each split?

Use the publisher's synthetic trace annotations as reference labels. Keep entire trace domains apart because the publisher uses one request template per domain. Take CLINC150's published out-of-scope rows only and keep its original test separate. The CLINC validation pool supplies two disjoint slices. The course selects nine of the trace dataset's failure categories plus clean and out-of-scope.

In [3]:
import gc
import importlib.metadata
import pandas as pd
from huggingface_hub import hf_hub_download
from sentence_transformers import SentenceTransformer
from transformers import AutoModel, AutoModelForCausalLM, AutoTokenizer

assert torch.cuda.is_available() and "T4" in torch.cuda.get_device_name()
device = torch.device("cuda")
torch.manual_seed(SEED)
torch.set_num_threads(4)
out = Path("results")
out.mkdir(exist_ok=True)

TRACE = "fahadhafeezofficial/agent-failure-bench"
TRACE_REV = "93257fb35460ed2916189467491e153d9f5844b2"
OOS = "clinc/clinc_oos"
OOS_REV = "155b9c710419136e17307b80d0a13e68cd46b4ec"
MODERN = "answerdotai/ModernBERT-base"
MODERN_REV = "8949b909ec900327062f0ebf497f51aef5e6f0c8"
MINILM = "sentence-transformers/all-MiniLM-L6-v2"
MINILM_REV = "1110a243fdf4706b3f48f1d95db1a4f5529b4d41"
LLM = "Qwen/Qwen2.5-0.5B-Instruct"
LLM_REV = "7ae557604adf67be50417f59c2c2f167def9a775"

def save(name, value):
    (out / name).write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n")

def stream_trace():
    for split in ("train", "validation", "test"):
        path = hf_hub_download(TRACE, f"dataset/{split}.jsonl", revision=TRACE_REV, repo_type="dataset")
        with open(path, encoding="utf-8") as stream:
            for line in stream:
                yield json.loads(line)

def read_oos(split):
    path = hf_hub_download(OOS, f"plus/{split}-00000-of-00001.parquet", revision=OOS_REV, repo_type="dataset")
    rows = pd.read_parquet(path)
    return list(rows.loc[rows["intent"] == 42, "text"])

oos_val = sorted(read_oos("validation"), key=lambda text: stable_id(norm(text)))
val_pool, calibration_pool = oos_val[::2], oos_val[1::2]
slices, domain_assignment = prepare_splits(stream_trace(), {
    "train": read_oos("train"), "gold_val": val_pool,
    "gold_calibration": calibration_pool, "gold_test": read_oos("test"),
})
coverage = {key: dict(Counter(r["label"] for r in rows)) for key, rows in slices.items()}
missing = {key: sorted(set(CLASSES) - set(counts)) for key, counts in coverage.items()}
save("split-audit.json", {"domains": domain_assignment, "counts": coverage, "missing": missing,
                        "ids": {k: [r["id"] for r in rows] for k, rows in slices.items()},
                        "exact_input_hashes": {k: [stable_id(norm(r["text"])) for r in rows] for k, rows in slices.items()},
                        "sources": {TRACE: TRACE_REV, OOS: OOS_REV}})
print("Class counts", coverage, "missing", missing)
assert not any(missing.values()), "A split lacks reference class coverage"
assert min(coverage["train"].values()) >= 8, "SetFit needs at least eight options per class"
all_rows = [r for rows in slices.values() for r in rows]
for key, values in {
    "ids": [r["id"] for r in all_rows],
    "groups": [r["group"] for r in all_rows],
    "normalised inputs": [norm(r["text"]) for r in all_rows],
}.items():
    assert len(values) == len(set(values)), f"Leakage in {key}"
labels = {key: label_ids(rows) for key, rows in slices.items()}
texts = {key: [r["text"] for r in rows] for key, rows in slices.items()}

Class counts {'train': {'clean': 32, 'invalid_tool_arguments': 32, 'incorrect_final_answer': 32, 'premature_completion': 32, 'out_of_scope': 32, 'failure_to_verify_completion': 32, 'failure_to_recover': 32, 'failure_to_follow_constraints': 32, 'incorrect_tool_selection': 32, 'missing_required_arguments': 32, 'hallucinated_tool_output': 32}, 'gold_val': {'clean': 12, 'invalid_tool_arguments': 12, 'failure_to_follow_constraints': 12, 'incorrect_tool_selection': 12, 'premature_completion': 12, 'out_of_scope': 12, 'hallucinated_tool_output': 12, 'incorrect_final_answer': 12, 'failure_to_verify_completion': 12, 'failure_to_recover': 12, 'missing_required_arguments': 12}, 'gold_calibration': {'clean': 12, 'out_of_scope': 12, 'missing_required_arguments': 12, 'invalid_tool_arguments': 12, 'incorrect_tool_selection': 12, 'failure_to_verify_completion': 12, 'incorrect_final_answer': 12, 'failure_to_follow_constraints': 12, 'failure_to_recover': 12, 'hallucinated_tool_output': 12, 'premature_com

## Where does the teacher disagree with the reference labels?

The local teacher learns lexical features and uses a deliberately incomplete rubric. It groups two adjacent failure categories with broader classes. Its probabilities still distinguish the remaining options. This controlled teacher makes an audit and a targeted repair possible without calling a commercial model.

In [4]:
teacher = teacher_fit(slices["train"])
teacher_targets = {key: teacher_probs(teacher, rows) for key, rows in slices.items()}
audit = per_class(labels["gold_val"], teacher_targets["gold_val"].argmax(-1))
override_names = [name for name in BLIND_SPOTS if audit["recall"][name] == 0.0]
assert override_names and set(override_names) == set(BLIND_SPOTS)
overrides = [CLASSES.index(name) for name in override_names]
save("teacher-audit.json", {"class_order": CLASSES, "val": audit, "hybrid_overrides": override_names})
print("Teacher validation audit", audit)

Teacher validation audit {'confusion': [[12, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 12, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 12, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 12, 0, 0], [0, 0, 0, 0, 12, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 12, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 12, 0, 0, 0, 0], [0, 0, 0, 0, 0, 12, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 12, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 12, 0], [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 11]], 'macro_f1': 0.7519851867677955, 'recall': {'incorrect_tool_selection': 1.0, 'invalid_tool_arguments': 1.0, 'missing_required_arguments': 1.0, 'hallucinated_tool_output': 0.0, 'failure_to_follow_constraints': 1.0, 'premature_completion': 1.0, 'failure_to_recover': 1.0, 'failure_to_verify_completion': 0.0, 'incorrect_final_answer': 1.0, 'clean': 1.0, 'out_of_scope': 0.9166666666666666}, 'counts': {'incorrect_tool_selection': 12, 'invalid_tool_arguments': 12, 'missing_required_arguments': 12, 'hallucinated_tool_output': 12, 'failure_to_follow_constraints': 12,

## Can a few examples teach the decision?

Use SetFit's two stages. Train the sentence embedder to bring same-class pairs closer and push different-class pairs apart, then fit a logistic classifier on the sampled sentences. Each option contributes the same labelled-example budget.

In [5]:
setfit_model = SentenceTransformer(MINILM, revision=MINILM_REV, device=str(device))
setfit_model.max_seq_length = MAX_LENGTH
setfit_head = tune_sentence_embedder(setfit_model, slices["train"], device)
setfit_model.half()
def setfit_batch(text):
    return setfit_head.predict_proba(setfit_model.encode(text, show_progress_bar=False))

setfit_test = setfit_batch(texts["gold_test"])

## How is milliseconds per trace measured?

Warm up each model, synchronise the device around every request, and time the same held-out rows in the same order at batch size one. Include tokenisation and classifier or option scoring. Exclude download and model loading. Compare with one forced-choice next-token decision from a generative checkpoint.

In [6]:
def measure_latency(read):
    roster = texts["gold_test"][:24]
    for text in roster[:4]:
        read(text)
    durations = []
    for text in roster:
        torch.cuda.synchronize()
        start = perf_counter()
        read(text)
        torch.cuda.synchronize()
        durations.append((perf_counter() - start) * 1000)
    ordered = sorted(durations)
    return {"n": len(roster), "ms": durations, "median_ms": float(np.median(durations)),
            "p95_nearest_rank_ms": ordered[math.ceil(0.95 * len(ordered)) - 1]}

latency = {"setfit": measure_latency(lambda text: setfit_batch([text]))}
del setfit_model
gc.collect()
torch.cuda.empty_cache()

## What changes between hard, soft and hybrid training?

A shared encoder embeds each trace and each option description. A two-layer head scores their interaction. Only the target distribution changes across these arms. The hybrid keeps gold labels where the audit found a teacher blind spot. It uses teacher probabilities on the other examples.

In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODERN, revision=MODERN_REV)
def new_encoder():
    return AutoModel.from_pretrained(MODERN, revision=MODERN_REV,
                                     attn_implementation="sdpa", reference_compile=False)

features = {key: encode_features(tokenizer, values) for key, values in texts.items()}
input_width = {key: [len(tokens) for tokens in tokenizer(values, truncation=False)["input_ids"]]
               for key, values in texts.items()}
save("input-budget.json", {"max_tokens": MAX_LENGTH, "side": "left",
                           "truncated": {key: sum(n > MAX_LENGTH for n in values)
                                         for key, values in input_width.items()},
                           "maximum": {key: max(values) for key, values in input_width.items()}})
options = device_features(encode_features(tokenizer, list(OPTIONS.values()), max_length=72), device)
gold = F.one_hot(labels["train"], num_classes=len(CLASSES)).float()
targets = {"hard_gold": gold, "soft_teacher": teacher_targets["train"],
           "hybrid": hybrid_targets(teacher_targets["train"], labels["train"], overrides)}
assert all(torch.allclose(values.sum(-1), torch.ones(len(values))) for values in targets.values())

## Which checkpoint would each selection rule keep?

At every epoch record agreement with the teacher and macro-F1 against the reference validation labels. Choose each encoder's main checkpoint by reference validation macro-F1. For the soft-target arm also select a checkpoint by teacher agreement and score it on the locked reference test. The rules may select the same epoch. Read their test errors even when they agree.

In [8]:
histories, scores, predictions = {}, {}, {}
scores["setfit"] = per_class(labels["gold_test"], setfit_test.argmax(-1))
predictions["setfit"] = setfit_test.tolist()
for name, arm_targets in targets.items():
    model, history, checkpoints = fit_options(
        new_encoder, features, options, labels, arm_targets,
        teacher_targets["gold_val"].argmax(-1), device)
    histories[name] = {"epochs": history, "chosen_gold_teacher": choose_checkpoints(history)}
    model.load_state_dict(checkpoints["gold"])
    test_logits = evaluate_options(model, features["gold_test"], options, device)
    scores[name] = per_class(labels["gold_test"], test_logits.argmax(-1))
    predictions[name] = test_logits.softmax(-1).tolist()
    if name == "soft_teacher":
        model.load_state_dict(checkpoints["teacher"])
        other = evaluate_options(model, features["gold_test"], options, device)
        scores["soft_selected_by_teacher"] = per_class(labels["gold_test"], other.argmax(-1))
        predictions["soft_selected_by_teacher"] = other.softmax(-1).tolist()
    if name == "hybrid":
        with torch.inference_mode():
            latency[name] = measure_latency(lambda text: evaluate_options(
                model, encode_features(tokenizer, [text]), options, device))
        calibration_logits = evaluate_options(model, features["gold_calibration"], options, device)
        temperature = fit_temperature(calibration_logits, labels["gold_calibration"])
        calibration = {"temperature": temperature, "test_before": reliability(test_logits, labels["gold_test"]),
                       "test_after": reliability(test_logits, labels["gold_test"], temperature),
                       "fit_count": len(calibration_logits)}
        assert torch.equal(test_logits.argmax(-1), (test_logits / temperature).argmax(-1))
        predictions["hybrid_scaled"] = (test_logits / temperature).softmax(-1).tolist()
    del model, checkpoints
    gc.collect()
    torch.cuda.empty_cache()

save("encoder-results.json", {"class_order": CLASSES, "histories": histories, "scores": scores,
                              "calibration": calibration, "reference": slices["gold_test"],
                              "probabilities": predictions})
print("Gold test macro-F1", {name: score["macro_f1"] for name, score in scores.items()})
print("Calibration", calibration)

Gold test macro-F1 {'setfit': 0.9713952038495649, 'hard_gold': 0.9885918003565063, 'soft_teacher': 0.6679732388823297, 'soft_selected_by_teacher': 0.6679732388823297, 'hybrid': 0.8787878787878788}
Calibration {'temperature': 0.17101559770838878, 'test_before': {'ece': 0.45207834887233644, 'nll': 0.9139265418052673, 'bins': [{'count': 0, 'low': 0.0, 'high': 0.1, 'confidence': None, 'accuracy': None}, {'count': 0, 'low': 0.1, 'high': 0.2, 'confidence': None, 'accuracy': None}, {'count': 54, 'low': 0.2, 'high': 0.3, 'confidence': 0.26731637120246887, 'accuracy': 0.7037037014961243}, {'count': 55, 'low': 0.3, 'high': 0.4, 'confidence': 0.3440318703651428, 'accuracy': 1.0}, {'count': 35, 'low': 0.4, 'high': 0.5, 'confidence': 0.4455528259277344, 'accuracy': 1.0}, {'count': 0, 'low': 0.5, 'high': 0.6, 'confidence': None, 'accuracy': None}, {'count': 0, 'low': 0.6, 'high': 0.7, 'confidence': None, 'accuracy': None}, {'count': 0, 'low': 0.7, 'high': 0.8, 'confidence': None, 'accuracy': None}, 

## How fast is a generative forced-choice comparison?

Score only the next-token options from the same request and option catalogue. There is no tool execution and no free-form generation. Report the generative baseline as a separate inference comparison.

In [9]:
llm_tokenizer = AutoTokenizer.from_pretrained(LLM, revision=LLM_REV)
llm = AutoModelForCausalLM.from_pretrained(LLM, revision=LLM_REV,
                                         dtype=torch.float16).to(device).eval()
letters = [chr(65 + i) for i in range(len(CLASSES))]
letter_ids = [llm_tokenizer.encode(letter, add_special_tokens=False) for letter in letters]
assert all(len(tokens) == 1 for tokens in letter_ids)
token_ids = [tokens[0] for tokens in letter_ids]
catalogue = "\n".join(f"{letter}. {description}" for letter, description in zip(letters, OPTIONS.values()))

@torch.inference_mode()
def llm_choose(text):
    # The raw trace is quoted data, and the only permissible output is an option token.
    trace_ids = llm_tokenizer.encode(text, add_special_tokens=False)[-MAX_LENGTH:]
    trace = llm_tokenizer.decode(trace_ids, skip_special_tokens=True)
    prompt = f"Classify this recorded trace or request. Choose one option letter.\n{catalogue}\nRecord: {trace}\nChoice:"
    values = llm_tokenizer(prompt, return_tensors="pt").to(device)
    return int(llm(**values, use_cache=False, logits_to_keep=1).logits[0, -1, token_ids].argmax())

latency["qwen_forced_choice"] = measure_latency(llm_choose)
scores["qwen_forced_choice_latency_roster"] = per_class(
    labels["gold_test"][:24], [llm_choose(text) for text in texts["gold_test"][:24]])
save("latency.json", {"hardware": torch.cuda.get_device_name(), "model_revisions": {
    MODERN: MODERN_REV, MINILM: MINILM_REV, LLM: LLM_REV},
    "packages": {name: importlib.metadata.version(name) for name in
                 ("torch", "transformers", "sentence-transformers", "scikit-learn", "pyarrow", "huggingface-hub")},
    "roster": [r["id"] for r in slices["gold_test"][:24]], "measurements": latency,
    "qwen_latency_roster_scores": scores["qwen_forced_choice_latency_roster"]})
print("Median milliseconds per trace", {name: stats["median_ms"] for name, stats in latency.items()})

Median milliseconds per trace {'setfit': 12.487147499996354, 'hybrid': 59.14070649998848, 'qwen_forced_choice': 31.16505299999517}


## Read the result

Check reference coverage before reading aggregate scores. Compare the teacher confusion matrix with the held-out gains or regressions for each student. Read the validation epoch choices separately from their test results. Temperature is fitted on reference calibration data, then evaluated on reference test.

[AgentFailureBench dataset card](https://huggingface.co/datasets/fahadhafeezofficial/agent-failure-bench) · [CLINC150](https://huggingface.co/datasets/clinc/clinc_oos) · [ModernBERT](https://arxiv.org/abs/2412.13663) · [SetFit](https://arxiv.org/abs/2209.11055) · [Calibration](https://arxiv.org/abs/1706.04599) · [Course lesson](https://docs.recohut.com/courses/advanced-llm-fine-tuning/3-adapt-models/12b-train-a-fast-encoder-decision-model).